## Pipeline: Tabela de Monitoramento do Modelo - ME BR v7

Este notebook consolida todas as informacoes em **duas tabelas** para monitoramento do modelo de credito.

---

### Tabelas Produzidas

| Tabela | Descricao | Comportamento de Ingestao |
|--------|-----------|---------------------------|
| `ds_catalog_dev.default.monitoring_me_br` | Consolidacao por cliente x safra | **MISTO**: INSERT novos + UPDATE targets |
| `ds_catalog_dev.default.monitoring_metrics_me_br` | Metricas agregadas por safra | **OVERWRITE**: recalculado a cada execucao |

---

### Month Shift e Alinhamento Temporal

As tabelas fonte operam em dois espacos temporais:

| Tabela | reference_month | Espaco |
|--------|----------------|--------|
| `abt_inference_me_br` | m-1 | Dados historicos |
| `portfolio_abt_group_me_br` | m-1 | Medianas do portfolio |
| `apply_model_me_br` | m | Mes de decisao |
| `targets_me_br` | m | Resultado observado |

A monitoring usa **reference_month = m** (mes de decisao) como eixo principal,
e inclui `feature_reference_month = m-1` para analises de estabilidade de features.

**Joins**:
```sql
apply_model <-> targets:    reference_month direto
apply_model <-> abt:        abt.reference_month = add_months(am.reference_month, -1)
apply_model <-> portfolio:  pag.reference_month = add_months(am.reference_month, -1)
```

**Uso das duas colunas temporais**:
- `reference_month`: PSI de scores, KS, Gini, performance do modelo
- `feature_reference_month`: PSI de features, drift detection, estabilidade

---

### Tabela 1: monitoring_me_br (dados por cliente x safra)

#### Comportamento: MISTO (INSERT + UPDATE Seletivo)

| Origem | Colunas | Comportamento na Monitoring |
|--------|---------|-----------------------------|
| `abt_inference` | Features, score, historical_weight, reference_value, reference_value_clp, payment_term | **Imutavel** |
| `portfolio_abt_group` | Medianas, portfolio_score | **Imutavel** |
| `apply_model` | adjusted_score | **Imutavel** |
| `targets` | percent7mob, billed, overdue | **Atualizavel** — retroativo 24m |

---

### Tabela 2: monitoring_metrics_me_br (metricas por safra)

Agrupa os dados de `monitoring_me_br` em **metricas de saude do modelo por reference_month**.
Recalculada a cada execucao (OVERWRITE) pois os targets atualizam retroativamente.

#### Organizacao das Metricas

| Grupo | Metricas | Disponibilidade |
|-------|----------|----------------|
| **Populacao** | total_clients, pop/pct por faixa (LOW/MEDIUM/HIGH) | Sempre |
| **Score** | avg_score, median_score | Sempre |
| **Limite USD** | avg_credit_limit, median_credit_limit, total_credit_limit | Sempre |
| **Limite CLP** | avg_credit_limit_clp, median_credit_limit_clp, total_credit_limit_clp | Sempre |
| **Financeiro** | total_billed_usd, overdue_usd, overdue_pct | Sempre |
| **Bad Rate** | bad_rate overall + por faixa | Quando target disponivel |
| **Lift** | lift por faixa | Quando target disponivel |
| **Discriminacao** | KS, ROC-AUC, Gini | Quando target tem ambas classes |
| **PSI** | psi_vs_base, psi_rolling + classificacao | Sempre |
| **Transicao** | pct melhorou/manteve/piorou | A partir da 2a safra |

#### Configuracao (alinhada com monitoring_report)
- **Score**: `integrated_score`
- **Target**: `target_percent7mob1`
- **Faixas**: 1-BAIXO (<=0.02), 2-MEDIO (<=0.30), 3-ALTO (>0.30)
- **PSI base**: primeira safra disponivel

---

### Pre-requisitos

Os 3 notebooks anteriores devem ter sido executados com sucesso:
- **NB1**: popula `abt_inference` e `portfolio_abt_group`
- **NB2**: popula `apply_model`
- **NB3**: popula `targets`

In [0]:
%sql
-- Tabela consolidada para monitoramento de saude do modelo
-- Reune features (NB1), scores (NB1+NB2), portfolio (NB1) e targets (NB3)
-- Colunas de targets sao atualizadas retroativamente; features/scores sao imutaveis
DROP TABLE IF EXISTS ds_catalog_dev.default.monitoring_me_br;
CREATE TABLE ds_catalog_dev.default.monitoring_me_br (
  -- === Colunas comuns ===
  id_customer                    INT,
  customer_name                  STRING,
  country                        STRING,
  reference_month                DATE,        -- Mes de decisao (m)
  feature_reference_month        DATE,        -- Mes dos dados (m-1)

  -- === Features (abt_inference_me_br) — imutaveis ===
  total_amount             DOUBLE,
  overdue_amount              DOUBLE,
  overdue_pct                DOUBLE,
  months_with_billing        INT,
  months_defaulted           INT,
  pct_months_overdue_10_20        DOUBLE,
  pct_months_overdue_20_30        DOUBLE,
  pct_months_overdue_30_50        DOUBLE,
  pct_months_overdue_50_plus        DOUBLE,
  flag_transacted            INT,

  -- === Scores individuais (abt_inference_me_br) — imutaveis ===
  score                          DOUBLE,
  historical_weight              DOUBLE,

  -- === Exposicao e prazo (abt_inference_me_br) — imutaveis ===
  reference_value                DOUBLE,      -- pico de exposicao 12m (USD)
  reference_value_clp            DOUBLE,      -- pico de exposicao 12m (CLP)
  payment_term                   DOUBLE,      -- prazo comercial medio (meses, clip [1,3])

  -- === Scores portfolio (portfolio_abt_group_me_br) — imutaveis ===
  median_cluster_10           DOUBLE,
  median_cluster_20           DOUBLE,
  median_cluster_30           DOUBLE,
  median_cluster_50           DOUBLE,
  portfolio_score                DOUBLE,

  -- === Score final (apply_model_me_br) — imutavel ===
  adjusted_score                 DOUBLE,
  score_band                     STRING,
  integrated_score               DOUBLE,
  integrated_score_band          STRING,
  credit_limit                   DOUBLE,
  credit_limit_clp               DOUBLE,
  credit_limit_end               DOUBLE,
  credit_limit_end_clp           DOUBLE,

  -- === Targets (targets_me_br) — atualizacao retroativa ===
  target_percent7mob1            INT,
  target_percent7mob3            INT,
  target_percent7mob6            INT,
  target_percent7mob9            INT,
  target_percent7mob12           INT,
  target_billed_1m               DOUBLE,
  target_overdue_1m              DOUBLE,
  target_overdue_pct_1m          DOUBLE,
  target_billed_3m               DOUBLE,
  target_overdue_3m              DOUBLE,
  target_overdue_pct_3m          DOUBLE,
  target_billed_6m               DOUBLE,
  target_overdue_6m              DOUBLE,
  target_overdue_pct_6m          DOUBLE,
  target_billed_12m              DOUBLE,
  target_overdue_12m             DOUBLE,
  target_overdue_pct_12m     DOUBLE,

  updated_at                     TIMESTAMP
)
USING DELTA
TBLPROPERTIES (
  'delta.autoOptimize.optimizeWrite' = 'true'
)

In [0]:
%sql
-- Valida que as 4 tabelas de entrada tem dados antes de prosseguir
SELECT 
  'abt_inference_me_br' AS tabela,
  COUNT(*) AS total_linhas,
  CAST(MIN(reference_month) AS STRING) AS safra_min,
  CAST(MAX(reference_month) AS STRING) AS safra_max
FROM ds_catalog_dev.default.abt_inference_me_br

UNION ALL

SELECT 
  'portfolio_abt_group_me_br',
  COUNT(*),
  CAST(MIN(reference_month) AS STRING),
  CAST(MAX(reference_month) AS STRING)
FROM ds_catalog_dev.default.portfolio_abt_group_me_br

UNION ALL

SELECT 
  'apply_model_me_br',
  COUNT(*),
  CAST(MIN(reference_month) AS STRING),
  CAST(MAX(reference_month) AS STRING)
FROM ds_catalog_dev.default.apply_model_me_br

UNION ALL

SELECT 
  'targets_me_br',
  COUNT(*),
  CAST(MIN(reference_month) AS STRING),
  CAST(MAX(reference_month) AS STRING)
FROM ds_catalog_dev.default.targets_me_br

tabela,total_linhas,safra_min,safra_max
abt_inference_me_br,83486,2020-01-01,2026-04-01
portfolio_abt_group_me_br,76,2020-01-01,2026-04-01
apply_model_me_br,83436,2020-02-01,2026-05-01
targets_me_br,83486,2020-02-01,2026-05-01


In [0]:
%sql
-- Base: apply_model (reference_month = m, mes de decisao)
-- Joins:
--   targets: direto em reference_month (ambos em espaco m)
--   abt_inference: shift -1 mes (abt esta em m-1)
--   portfolio_abt_group: shift -1 mes (portfolio esta em m-1)
-- feature_reference_month: preserva o mes original dos dados (m-1)
CREATE OR REPLACE TEMP VIEW monitoring_source AS
SELECT 
  -- Colunas comuns (de apply_model como tabela base)
  am.id_customer,
  am.customer_name,
  am.country,
  am.reference_month,
  add_months(am.reference_month, -1) AS feature_reference_month,

  -- Features (abt_inference, espaco m-1)
  inf.total_amount,
  inf.overdue_amount,
  inf.overdue_pct,
  inf.months_with_billing,
  inf.months_defaulted,
  inf.pct_months_overdue_10_20,
  inf.pct_months_overdue_20_30,
  inf.pct_months_overdue_30_50,
  inf.pct_months_overdue_50_plus,
  inf.flag_transacted,

  -- Scores individuais (abt_inference, espaco m-1)
  inf.score,
  inf.historical_weight,

  -- Exposicao e prazo (abt_inference, espaco m-1) — imutaveis
  COALESCE(inf.reference_value, 0)     AS reference_value,
  COALESCE(inf.reference_value_clp, 0) AS reference_value_clp,
  COALESCE(inf.payment_term, 1.0)      AS payment_term,

  -- Scores portfolio (portfolio_abt_group, espaco m-1)
  pag.median_cluster_10,
  pag.median_cluster_20,
  pag.median_cluster_30,
  pag.median_cluster_50,
  pag.portfolio_score,

  -- Score final (apply_model, espaco m)
  am.adjusted_score,
  am.score_band,
  am.integrated_score,
  am.integrated_score_band,
  am.credit_limit,
  am.credit_limit_clp,
  am.credit_limit_end,
  am.credit_limit_end_clp,

  -- Targets (backward-updating, espaco m) — prefixo target_
  t.percent7mob1       AS target_percent7mob1,
  t.percent7mob3       AS target_percent7mob3,
  t.percent7mob6       AS target_percent7mob6,
  t.percent7mob9       AS target_percent7mob9,
  t.percent7mob12      AS target_percent7mob12,
  t.billed_1m          AS target_billed_1m,
  t.overdue_1m         AS target_overdue_1m,
  t.overdue_pct_1m     AS target_overdue_pct_1m,
  t.billed_3m          AS target_billed_3m,
  t.overdue_3m         AS target_overdue_3m,
  t.overdue_pct_3m     AS target_overdue_pct_3m,
  t.billed_6m          AS target_billed_6m,
  t.overdue_6m         AS target_overdue_6m,
  t.overdue_pct_6m     AS target_overdue_pct_6m,
  t.billed_12m         AS target_billed_12m,
  t.overdue_12m        AS target_overdue_12m,
  t.overdue_pct_12m    AS target_overdue_pct_12m,

  current_timestamp() AS updated_at

FROM ds_catalog_dev.default.apply_model_me_br AS am
LEFT JOIN ds_catalog_dev.default.abt_inference_me_br AS inf
  ON inf.id_customer = am.id_customer
 AND inf.reference_month = add_months(am.reference_month, -1)
LEFT JOIN ds_catalog_dev.default.portfolio_abt_group_me_br AS pag
  ON pag.reference_month = add_months(am.reference_month, -1)
LEFT JOIN ds_catalog_dev.default.targets_me_br AS t
  ON t.id_customer = am.id_customer
 AND t.reference_month = am.reference_month
WHERE am.reference_month >= LEAST(
  add_months(date_trunc('month', current_date()), -24),
  COALESCE(
    (SELECT MAX(reference_month) FROM ds_catalog_dev.default.monitoring_me_br),
    DATE '1900-01-01'
  )
)

In [0]:
%sql
-- Upsert com atualizacao seletiva:
--   NOT MATCHED: insere todas as colunas (features + scores + targets)
--   MATCHED: atualiza APENAS colunas de targets (backward-updating)
--            features, scores e feature_reference_month sao imutaveis
MERGE INTO ds_catalog_dev.default.monitoring_me_br AS target
USING monitoring_source AS source
ON target.reference_month = source.reference_month 
   AND target.id_customer = source.id_customer
WHEN MATCHED THEN UPDATE SET
  target.integrated_score        = source.integrated_score,
  target.integrated_score_band   = source.integrated_score_band,
  target.credit_limit            = source.credit_limit,
  target.credit_limit_clp        = source.credit_limit_clp,
  target.credit_limit_end        = source.credit_limit_end,
  target.credit_limit_end_clp    = source.credit_limit_end_clp,
  target.target_percent7mob1     = source.target_percent7mob1,
  target.target_percent7mob3     = source.target_percent7mob3,
  target.target_percent7mob6     = source.target_percent7mob6,
  target.target_percent7mob9     = source.target_percent7mob9,
  target.target_percent7mob12    = source.target_percent7mob12,
  target.target_billed_1m        = source.target_billed_1m,
  target.target_overdue_1m       = source.target_overdue_1m,
  target.target_overdue_pct_1m   = source.target_overdue_pct_1m,
  target.target_billed_3m        = source.target_billed_3m,
  target.target_overdue_3m       = source.target_overdue_3m,
  target.target_overdue_pct_3m   = source.target_overdue_pct_3m,
  target.target_billed_6m        = source.target_billed_6m,
  target.target_overdue_6m       = source.target_overdue_6m,
  target.target_overdue_pct_6m   = source.target_overdue_pct_6m,
  target.target_billed_12m       = source.target_billed_12m,
  target.target_overdue_12m      = source.target_overdue_12m,
  target.target_overdue_pct_12m  = source.target_overdue_pct_12m,
  target.updated_at              = source.updated_at
WHEN NOT MATCHED THEN INSERT *

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
83436,0,0,83436


In [0]:
%sql
select * from monitoring_source

id_customer,customer_name,country,reference_month,feature_reference_month,total_amount,overdue_amount,overdue_pct,months_with_billing,months_defaulted,pct_months_overdue_10_20,pct_months_overdue_20_30,pct_months_overdue_30_50,pct_months_overdue_50_plus,flag_transacted,score,historical_weight,reference_value,reference_value_clp,payment_term,median_cluster_10,median_cluster_20,median_cluster_30,median_cluster_50,portfolio_score,adjusted_score,score_band,integrated_score,integrated_score_band,credit_limit,credit_limit_clp,credit_limit_end,credit_limit_end_clp,target_percent7mob1,target_percent7mob3,target_percent7mob6,target_percent7mob9,target_percent7mob12,target_billed_1m,target_overdue_1m,target_overdue_pct_1m,target_billed_3m,target_overdue_3m,target_overdue_pct_3m,target_billed_6m,target_overdue_6m,target_overdue_pct_6m,target_billed_12m,target_overdue_12m,target_overdue_pct_12m,updated_at
252372,SOCIEDAD COMERCIAL MALLORCA LTDA.,CHILE,2026-04-01,2026-03-01,952126.74,26801.78,2.81,5,0,0.0,0.0,0.0,0.0,0,0.0,1.0,197796.8393,1.7907734353E8,1.3544,13.63,23.9,39.77,74.87,0.4414,0.0,1-BAIXO,0.0,1-BAIXO,525522.6562,4.806622739847E8,525522.6562,4.806622739847E8,null,null,null,null,null,0.0,0.0,null,0.0,0.0,null,0.0,0.0,null,0.0,0.0,null,2026-05-07T12:14:20.401+0000
95524,CONSORCIO INDUSTRIAL DE ALIMENTOS S.A.,CHILE,2025-11-01,2025-10-01,2671841.2,66734.23,2.5,7,0,0.0,0.0,0.0,0.0,1,0.0,1.0,316759.845,2.8678168863E8,1.5667,14.77,25.23,40.03,79.13,0.5218,0.0,1-BAIXO,0.0,1-BAIXO,982040.2132,9.343377722938E8,982040.2132,9.343377722938E8,1,1,1,1,1,2872.54,2872.54,1.0,316160.39999999997,114782.13,0.3631,457329.68,115614.93000000001,0.2528,457329.68,115614.93000000001,0.2528,2026-05-07T12:14:20.401+0000
279803,SODEXO CHILE SPA,CHILE,2024-12-01,2024-11-01,2356023.88,596964.48,25.34,6,2,0.0,0.0,0.0,33.33,1,0.277906,1.0,488957.5617,4.426826109E8,2.0048,14.03,24.85,38.8,83.38,0.5974,0.277906,2-MEDIO,0.277906,2-MEDIO,1306043.101,1.2709832937852E9,1306043.101,1.2709832937852E9,null,1,1,1,1,0.0,0.0,null,591997.3500000001,303610.43,0.5129,1427319.1700000002,1138932.25,0.798,2212434.7600000002,1924047.84,0.8697,2026-05-07T12:14:20.401+0000
6494,COMERCIAL DISER LIMITADA,CHILE,2025-08-01,2025-07-01,3667997.77,1157962.77,31.57,9,6,0.0,0.0,22.22,44.44,1,0.407182,1.0,330826.6679,2.995172272E8,1.3987,13.21,24.74,36.87,73.19,0.5096,0.407182,3-ALTO,0.407182,3-ALTO,800095.3009,7.654422529182E8,800095.3009,7.654422529182E8,null,1,1,1,1,0.0,0.0,null,1487882.2200000002,387664.85000000003,0.2605,1689844.08,393243.63000000006,0.2327,2268086.0500000003,393884.36000000004,0.1737,2026-05-07T12:14:20.401+0000
6494,COMERCIAL DISER LIMITADA,CHILE,2025-12-01,2025-11-01,4673458.91,1737762.53,37.18,8,7,0.0,12.5,25.0,50.0,0,0.521788,1.0,391867.3289,3.5478099915E8,1.3556,14.04,24.05,38.37,79.16,0.5029,0.522825,3-ALTO,0.522825,3-ALTO,1091230.3094,1.0284279442191E9,1091230.3094,1.0284279442191E9,null,0,0,0,0,0.0,0.0,null,226535.53999999998,6219.51,0.0275,780203.8300000001,6219.51,0.008,780203.8300000001,6219.51,0.008,2026-05-07T12:14:20.401+0000
538839,COMERCIAL HUALLE LIMITADA,CHILE,2025-06-01,2025-05-01,456260.03,456260.03,100.0,1,1,0.0,0.0,0.0,100.0,0,0.7075,0.3333,456260.03,4.1307957408E8,1.0,13.72,25.59,36.23,70.75,0.5035,0.57116,3-ALTO,0.57116,3-ALTO,740788.9256,6.914309734919E8,740788.9256,6.914309734919E8,0,0,0,1,1,435080.28,12265.14,0.0282,435080.28,12265.14,0.0282,435080.28,12265.14,0.0282,782381.13,190562.91999999998,0.2436,2026-05-07T12:14:20.401+0000
390455,COMERCIAL CATERBEEF CIA LTDA,CHILE,2026-04-01,2026-03-01,3237951.99,729463.88,22.53,3,2,0.0,0.0,66.67,0.0,0,0.265147,1.0,940680.07,8.516540944E8,1.3508,13.63,23.9,39.77,74.87,0.4414,0.261346,2-MEDIO,0.261346,2-MEDIO,1798737.8633,1.6460488738939E9,1798737.8633,1.6460488738939E9,null,null,null,null,null,0.0,0.0,null,0.0,0.0,null,0.0,0.0,null,0.0,0.0,null,2026-05-07T12:14:20.401+0000
142017,AGROSUPER COMERCIALIZADORA DE ALIMENTOS LIMITADA,CHILE,2026-04-01,2026-03-01,3607679.73,681571.54,18.89,6,2,0.0,0.0,0.0,33.

In [0]:
%sql
-- ============================================
-- SANITY CHECKS: monitoring_me_br
-- ============================================
SELECT 
  (SELECT COUNT(*) FROM ds_catalog_dev.default.monitoring_me_br) AS total_linhas,
  (SELECT MIN(reference_month) FROM ds_catalog_dev.default.monitoring_me_br) AS safra_min,
  (SELECT MAX(reference_month) FROM ds_catalog_dev.default.monitoring_me_br) AS safra_max,
  (SELECT COUNT(DISTINCT reference_month) FROM ds_catalog_dev.default.monitoring_me_br) AS total_safras,
  (SELECT COUNT(DISTINCT id_customer) FROM ds_catalog_dev.default.monitoring_me_br) AS total_clientes,
  -- Duplicatas
  (SELECT COUNT(*) FROM (
    SELECT reference_month, id_customer, COUNT(*) AS cnt
    FROM ds_catalog_dev.default.monitoring_me_br
    GROUP BY reference_month, id_customer HAVING cnt > 1
  )) AS duplicatas,
  -- Verificacao do month shift: monitoring.max deve = apply_model.max
  (SELECT MAX(reference_month) FROM ds_catalog_dev.default.apply_model_me_br) AS apply_model_max,
  -- Verificacao feature_reference_month = reference_month - 1
  (SELECT COUNT(*) FROM ds_catalog_dev.default.monitoring_me_br
   WHERE feature_reference_month != add_months(reference_month, -1)
  ) AS feature_month_inconsistencias,
  -- Completude: linhas sem score (abt_inference faltou)
  (SELECT COUNT(*) FROM ds_catalog_dev.default.monitoring_me_br WHERE score IS NULL) AS linhas_sem_score,
  -- Completude: linhas sem adjusted_score (apply_model faltou)
  (SELECT COUNT(*) FROM ds_catalog_dev.default.monitoring_me_br WHERE adjusted_score IS NULL) AS linhas_sem_adjusted_score,
  -- Completude: linhas sem targets (esperado para safras recentes)
  (SELECT COUNT(*) FROM ds_catalog_dev.default.monitoring_me_br WHERE target_percent7mob1 IS NULL AND target_percent7mob12 IS NULL) AS linhas_sem_targets

total_linhas,safra_min,safra_max,total_safras,total_clientes,duplicatas,apply_model_max,feature_month_inconsistencias,linhas_sem_score,linhas_sem_adjusted_score,linhas_sem_targets
83436,2020-02-01,2026-05-01,76,2866,0,2026-05-01,0,0,0,29527


## Tabela 2: Metricas Agregadas de Saude do Modelo

Calcula metricas por `reference_month` a partir de `monitoring_me_br`.

**Comportamento**: OVERWRITE completo a cada execucao (targets atualizam retroativamente).

**Grupos de metricas**:
1. Populacao e distribuicao por faixa de risco
2. Distribuicao de scores
3. Limite de credito (avg, median, total)
4. Volume financeiro (faturado e atraso em USD)
5. Bad rate (geral e por faixa)
6. Lift por faixa de risco
7. Discriminacao: KS, ROC-AUC, Gini
8. PSI vs base + rolling
9. Transicao entre faixas (melhorou/manteve/piorou)

**Configuracao** (alinhada com `monitoring_report_me_br`):
- Score: `integrated_score` | Target: `target_percent7mob1`
- Faixas: 1-BAIXO (<=0.02) | 2-MEDIO (<=0.30) | 3-ALTO (>0.30)
- PSI base: primeira safra disponivel

In [0]:
%sql
DROP TABLE IF EXISTS ds_catalog_dev.default.monitoring_metrics_me_br;
CREATE TABLE ds_catalog_dev.default.monitoring_metrics_me_br (
  reference_month                DATE,

  -- === Populacao e distribuicao ===
  total_clients                  INT,
  pop_baixo                      INT,
  pop_medio                      INT,
  pop_alto                       INT,
  pct_baixo                      DOUBLE,
  pct_medio                      DOUBLE,
  pct_alto                       DOUBLE,

  -- === Score ===
  avg_score                      DOUBLE,
  median_score                   DOUBLE,

  -- === Limite de credito USD ===
  avg_credit_limit               DOUBLE,
  median_credit_limit            DOUBLE,
  total_credit_limit             DOUBLE,

  -- === Limite de credito CLP ===
  avg_credit_limit_clp           DOUBLE,
  median_credit_limit_clp        DOUBLE,
  total_credit_limit_clp         DOUBLE,

  -- === Financeiro ===
  total_billed_usd               DOUBLE,
  overdue_usd                    DOUBLE,
  overdue_pct                    DOUBLE,

  -- === Bad rate ===
  bad_rate_overall               DOUBLE,
  bad_rate_baixo                 DOUBLE,
  bad_rate_medio                 DOUBLE,
  bad_rate_alto                  DOUBLE,

  -- === Lift ===
  lift_baixo                     DOUBLE,
  lift_medio                     DOUBLE,
  lift_alto                      DOUBLE,

  -- === Discriminacao ===
  ks                             DOUBLE,
  roc_auc                        DOUBLE,
  gini                           DOUBLE,

  -- === PSI ===
  psi_vs_base                    DOUBLE,
  psi_vs_base_classification     STRING,
  psi_rolling                    DOUBLE,
  psi_rolling_classification     STRING,

  -- === Transicao ===
  pct_improved                   DOUBLE,
  pct_maintained                 DOUBLE,
  pct_worsened                   DOUBLE,

  updated_at                     TIMESTAMP
)
USING DELTA
TBLPROPERTIES (
  'delta.autoOptimize.optimizeWrite' = 'true'
)

In [0]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from scipy.stats import ks_2samp
from pyspark.sql.types import (
    StructType, StructField, DateType, IntegerType, DoubleType, StringType, TimestampType
)
from datetime import datetime

# ====== CONFIGURACAO (alinhada com monitoring_report_me_br) ======
TARGET_COL = 'target_percent7mob1'
SCORE_COL = 'integrated_score'
BAND_COL = 'integrated_score_band'
BAND_NAMES = ['1-BAIXO', '2-MEDIO', '3-ALTO']
BAND_MAP = {'1-BAIXO': 1, '2-MEDIO': 2, '3-ALTO': 3}

# ====== FUNCOES AUXILIARES ======
def calc_psi(base_dist, current_dist, epsilon=1e-4):
    base = np.array(base_dist, dtype=float) + epsilon
    curr = np.array(current_dist, dtype=float) + epsilon
    base = base / base.sum()
    curr = curr / curr.sum()
    return round(float(np.sum((curr - base) * np.log(curr / base))), 6)

def classify_psi(val):
    if val < 0.10: return 'Estavel'
    elif val < 0.25: return 'Moderado'
    return 'Significativo'

def safe_round(val, decimals=6):
    return round(val, decimals) if val is not None and not (isinstance(val, float) and np.isnan(val)) else None

# ====== CARGA DE DADOS ======
df = spark.sql("""
    SELECT id_customer, reference_month, integrated_score, integrated_score_band,
           total_amount, overdue_amount,
           credit_limit_end, credit_limit_end_clp,
           target_percent7mob1
    FROM ds_catalog_dev.default.monitoring_me_br
    WHERE integrated_score IS NOT NULL
""").toPandas()

df = df.sort_values('reference_month')
months = sorted(df['reference_month'].unique())

print(f"Dados carregados: {len(df):,} registros, {len(months)} safras")
print(f"Range: {months[0]} a {months[-1]}")

# ====== DISTRIBUICOES PARA PSI ======
dist_map = {}
for m in months:
    vc = df[df['reference_month'] == m][BAND_COL].value_counts(normalize=True)
    dist_map[m] = [vc.get(b, 0) for b in BAND_NAMES]

base_dist = dist_map[months[0]]

# ====== CALCULO DE METRICAS POR SAFRA ======
rows = []
for i, m in enumerate(months):
    dm = df[df['reference_month'] == m]
    total = len(dm)

    # --- Populacao e distribuicao ---
    pop_baixo  = int((dm[BAND_COL] == '1-BAIXO').sum())
    pop_medio  = int((dm[BAND_COL] == '2-MEDIO').sum())
    pop_alto   = int((dm[BAND_COL] == '3-ALTO').sum())
    pct_baixo  = round(pop_baixo  / total * 100, 4) if total > 0 else None
    pct_medio  = round(pop_medio  / total * 100, 4) if total > 0 else None
    pct_alto   = round(pop_alto   / total * 100, 4) if total > 0 else None

    # --- Score ---
    avg_score = safe_round(dm[SCORE_COL].mean())
    median_score = safe_round(dm[SCORE_COL].median())

    # --- Limite de credito USD ---
    avg_credit_limit = safe_round(dm['credit_limit_end'].mean(), 2)
    median_credit_limit = safe_round(dm['credit_limit_end'].median(), 2)
    total_credit_limit = round(dm['credit_limit_end'].sum(), 2)

    # --- Limite de credito CLP ---
    avg_credit_limit_clp = safe_round(dm['credit_limit_end_clp'].mean(), 2)
    median_credit_limit_clp = safe_round(dm['credit_limit_end_clp'].median(), 2)
    total_credit_limit_clp = round(dm['credit_limit_end_clp'].sum(), 2)

    # --- Financeiro ---
    total_billed_usd = round(dm['total_amount'].sum(), 2)
    overdue_usd = round(dm['overdue_amount'].sum(), 2)
    overdue_pct = safe_round(overdue_usd / total_billed_usd * 100, 4) if total_billed_usd > 0 else None

    # --- Bad rate, Lift e Discriminacao (requer target) ---
    bad_rate_overall = bad_rate_baixo = bad_rate_medio = bad_rate_alto = None
    lift_baixo = lift_medio = lift_alto = None
    ks = roc_auc = gini_val = None

    dm_t = dm[dm[TARGET_COL].notna()].copy()
    if len(dm_t) > 0:
        dm_t[TARGET_COL] = dm_t[TARGET_COL].astype(int)
        y_true = dm_t[TARGET_COL].values
        y_score = dm_t[SCORE_COL].values

        bad_rate_overall = safe_round(y_true.mean())

        for band, attr in [('1-BAIXO', 'bad_rate_baixo'), ('2-MEDIO', 'bad_rate_medio'), ('3-ALTO', 'bad_rate_alto')]:
            mask = dm_t[BAND_COL] == band
            if mask.any():
                locals()[attr] = safe_round(dm_t.loc[mask, TARGET_COL].mean())

        # Lift
        if bad_rate_overall and bad_rate_overall > 0:
            lift_baixo = safe_round(bad_rate_baixo / bad_rate_overall, 4) if bad_rate_baixo is not None else None
            lift_medio = safe_round(bad_rate_medio / bad_rate_overall, 4) if bad_rate_medio is not None else None
            lift_alto  = safe_round(bad_rate_alto  / bad_rate_overall, 4) if bad_rate_alto  is not None else None

        # KS, AUC, Gini (precisa de ambas as classes)
        if y_true.sum() > 0 and y_true.sum() < len(y_true):
            good_scores = y_score[y_true == 0]
            bad_scores = y_score[y_true == 1]
            ks_stat, _ = ks_2samp(good_scores, bad_scores)
            ks = safe_round(ks_stat)
            auc_val = roc_auc_score(y_true, y_score)
            roc_auc = safe_round(auc_val)
            gini_val = safe_round(2 * auc_val - 1)

    # --- PSI ---
    psi_vs_base = calc_psi(base_dist, dist_map[m])
    psi_vs_base_class = classify_psi(psi_vs_base)
    psi_rolling = calc_psi(dist_map[months[i-1]], dist_map[m]) if i > 0 else None
    psi_rolling_class = classify_psi(psi_rolling) if psi_rolling is not None else None

    # --- Transicao (vs safra anterior) ---
    pct_improved = pct_maintained = pct_worsened = None
    if i > 0:
        prev_m = months[i - 1]
        df_prev = df[df['reference_month'] == prev_m][['id_customer', BAND_COL]].rename(
            columns={BAND_COL: 'prev_band'}
        )
        df_curr = dm[['id_customer', BAND_COL]].rename(
            columns={BAND_COL: 'curr_band'}
        )
        paired = df_prev.merge(df_curr, on='id_customer', how='inner')
        if len(paired) > 0:
            direction = paired['curr_band'].map(BAND_MAP) - paired['prev_band'].map(BAND_MAP)
            pct_improved = safe_round((direction < 0).mean() * 100, 4)
            pct_maintained = safe_round((direction == 0).mean() * 100, 4)
            pct_worsened = safe_round((direction > 0).mean() * 100, 4)

    rows.append({
        'reference_month': m,
        'total_clients': total,
        'pop_baixo': pop_baixo, 'pop_medio': pop_medio, 'pop_alto': pop_alto,
        'pct_baixo': pct_baixo, 'pct_medio': pct_medio, 'pct_alto': pct_alto,
        'avg_score': avg_score, 'median_score': median_score,
        'avg_credit_limit': avg_credit_limit, 'median_credit_limit': median_credit_limit, 'total_credit_limit': total_credit_limit,
        'avg_credit_limit_clp': avg_credit_limit_clp, 'median_credit_limit_clp': median_credit_limit_clp, 'total_credit_limit_clp': total_credit_limit_clp,
        'total_billed_usd': total_billed_usd, 'overdue_usd': overdue_usd, 'overdue_pct': overdue_pct,
        'bad_rate_overall': bad_rate_overall,
        'bad_rate_baixo': bad_rate_baixo, 'bad_rate_medio': bad_rate_medio, 'bad_rate_alto': bad_rate_alto,
        'lift_baixo': lift_baixo, 'lift_medio': lift_medio, 'lift_alto': lift_alto,
        'ks': ks, 'roc_auc': roc_auc, 'gini': gini_val,
        'psi_vs_base': psi_vs_base, 'psi_vs_base_classification': psi_vs_base_class,
        'psi_rolling': psi_rolling, 'psi_rolling_classification': psi_rolling_class,
        'pct_improved': pct_improved, 'pct_maintained': pct_maintained, 'pct_worsened': pct_worsened,
    })

df_metrics = pd.DataFrame(rows)
df_metrics['updated_at'] = datetime.now()

print(f"\nMetricas calculadas: {len(df_metrics)} safras")
print(f"Safras com bad_rate: {df_metrics['bad_rate_overall'].notna().sum()}")
print(f"Safras com KS/AUC/Gini: {df_metrics['ks'].notna().sum()}")
print(f"Safras com PSI rolling: {df_metrics['psi_rolling'].notna().sum()}")
print(f"Safras com transicao: {df_metrics['pct_improved'].notna().sum()}")

# ====== ESCRITA (OVERWRITE) ======
schema = StructType([
    StructField('reference_month', DateType()),
    StructField('total_clients', IntegerType()),
    StructField('pop_baixo', IntegerType()),
    StructField('pop_medio', IntegerType()),
    StructField('pop_alto', IntegerType()),
    StructField('pct_baixo', DoubleType()),
    StructField('pct_medio', DoubleType()),
    StructField('pct_alto', DoubleType()),
    StructField('avg_score', DoubleType()),
    StructField('median_score', DoubleType()),
    StructField('avg_credit_limit', DoubleType()),
    StructField('median_credit_limit', DoubleType()),
    StructField('total_credit_limit', DoubleType()),
    StructField('avg_credit_limit_clp', DoubleType()),
    StructField('median_credit_limit_clp', DoubleType()),
    StructField('total_credit_limit_clp', DoubleType()),
    StructField('total_billed_usd', DoubleType()),
    StructField('overdue_usd', DoubleType()),
    StructField('overdue_pct', DoubleType()),
    StructField('bad_rate_overall', DoubleType()),
    StructField('bad_rate_baixo', DoubleType()),
    StructField('bad_rate_medio', DoubleType()),
    StructField('bad_rate_alto', DoubleType()),
    StructField('lift_baixo', DoubleType()),
    StructField('lift_medio', DoubleType()),
    StructField('lift_alto', DoubleType()),
    StructField('ks', DoubleType()),
    StructField('roc_auc', DoubleType()),
    StructField('gini', DoubleType()),
    StructField('psi_vs_base', DoubleType()),
    StructField('psi_vs_base_classification', StringType()),
    StructField('psi_rolling', DoubleType()),
    StructField('psi_rolling_classification', StringType()),
    StructField('pct_improved', DoubleType()),
    StructField('pct_maintained', DoubleType()),
    StructField('pct_worsened', DoubleType()),
    StructField('updated_at', TimestampType()),
])

sdf = spark.createDataFrame(df_metrics, schema=schema)
sdf.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable('ds_catalog_dev.default.monitoring_metrics_me_br')

print(f"\nTabela ds_catalog_dev.default.monitoring_metrics_me_br gravada com {len(df_metrics)} linhas (OVERWRITE)")

Dados carregados: 83,436 registros, 76 safras
Range: 2020-02-01 a 2026-05-01

Metricas calculadas: 76 safras
Safras com bad_rate: 75
Safras com KS/AUC/Gini: 75
Safras com PSI rolling: 75
Safras com transicao: 75

Tabela teste.monitoring_metrics_me_br gravada com 76 linhas (OVERWRITE)


In [0]:
%sql
-- ============================================
-- SANITY CHECKS: monitoring_metrics_me_br
-- ============================================
SELECT 
  (SELECT COUNT(*) FROM ds_catalog_dev.default.monitoring_metrics_me_br) AS total_safras,
  (SELECT MIN(reference_month) FROM ds_catalog_dev.default.monitoring_metrics_me_br) AS safra_min,
  (SELECT MAX(reference_month) FROM ds_catalog_dev.default.monitoring_metrics_me_br) AS safra_max,
  -- Verificar que todas as safras de monitoring estao presentes
  (SELECT COUNT(DISTINCT reference_month) FROM ds_catalog_dev.default.monitoring_me_br
   WHERE integrated_score IS NOT NULL
  ) AS expected_safras,
  -- Metricas de discriminacao disponiveis
  (SELECT COUNT(*) FROM ds_catalog_dev.default.monitoring_metrics_me_br WHERE ks IS NOT NULL) AS safras_com_ks,
  (SELECT COUNT(*) FROM ds_catalog_dev.default.monitoring_metrics_me_br WHERE bad_rate_overall IS NOT NULL) AS safras_com_bad_rate,
  -- PSI ranges
  (SELECT ROUND(MAX(psi_vs_base), 4) FROM ds_catalog_dev.default.monitoring_metrics_me_br) AS max_psi_vs_base,
  (SELECT SUM(CASE WHEN psi_vs_base_classification = 'Significativo' THEN 1 ELSE 0 END) FROM ds_catalog_dev.default.monitoring_metrics_me_br) AS safras_psi_significativo,
  -- Transicao
  (SELECT COUNT(*) FROM ds_catalog_dev.default.monitoring_metrics_me_br WHERE pct_improved IS NOT NULL) AS safras_com_transicao,
  -- Consistencia: pct_baixo + pct_medio + pct_alto ~ 100
  (SELECT COUNT(*) FROM ds_catalog_dev.default.monitoring_metrics_me_br
   WHERE ABS(pct_baixo + pct_medio + pct_alto - 100.0) > 0.01
  ) AS inconsistencias_pct_faixas

total_safras,safra_min,safra_max,expected_safras,safras_com_ks,safras_com_bad_rate,max_psi_vs_base,safras_psi_significativo,safras_com_transicao,inconsistencias_pct_faixas
76,2020-02-01,2026-05-01,76,75,75,5.6244,74,75,0
